# Fairness Mitigation Experiments

This notebook compares controlled alternatives to the original income-threshold baseline. It does not replace the baseline or change prior audit outputs. The target is income above $50,000, not insurance eligibility.

Experiments include: training-only mutual-information screening for potential proxies; removing `RAC1P` and `SEX`; removing those attributes plus the top proxy candidates; separate group-label reweighting experiments for race and sex; held-out performance and fairness comparisons; SHAP contribution direction summaries; and counterfactual flip rates where protected features remain model inputs.

Proxy screening is associative and univariate; it does not establish causation or prove proxy discrimination. Group-label reweighting is an experimental mitigation, not a guarantee of fairness. Interpret all metrics with sample sizes and performance trade-offs.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.append(os.path.abspath(".."))
from src.audit.mitigation_experiments import run_fairness_mitigation_experiments

ROOT = Path("..").resolve()
RESULTS_DIR = ROOT / "results"
result = run_fairness_mitigation_experiments(
    processed_dir=str(ROOT / "data" / "processed"),
    baseline_model_path=str(RESULTS_DIR / "baseline_xgb.json"),
    output_path=str(RESULTS_DIR / "fairness_mitigation_experiments.json"),
)
print("Saved comparison to results/fairness_mitigation_experiments.json")

Saved comparison to results/fairness_mitigation_experiments.json


In [ ]:
proxy_rows = []
for protected, rows in result["proxy_associations"].items():
    for row in rows:
        proxy_rows.append({"protected_attribute": protected, **row})
display(pd.DataFrame(proxy_rows).sort_values(
    ["protected_attribute", "mutual_information"], ascending=[True, False]
))
print("Candidates removed in the proxy-ablation model:", result["proxy_candidates_removed"])

In [ ]:
summary_rows = []
for name, item in result["variants"].items():
    row = {
        "variant": name,
        "accuracy": item["performance"]["accuracy"],
        "F1": item["performance"]["f1"],
        "ROC_AUC": item["performance"]["roc_auc"],
        "fairness_risk": item["fairness_risk"],
        "fairness_status": item["fairness_status"],
        "RAC1P_DPD": item["fairness"]["RAC1P"]["demographic_parity_difference"],
        "RAC1P_DIR": item["fairness"]["RAC1P"]["disparate_impact_ratio"],
        "RAC1P_equal_opportunity_gap": item["fairness"]["RAC1P"]["equal_opportunity_difference"],
        "RAC1P_FPR_gap": item["fairness"]["RAC1P"]["equalized_odds_fpr_difference"],
        "SEX_DPD": item["fairness"]["SEX"]["demographic_parity_difference"],
        "SEX_DIR": item["fairness"]["SEX"]["disparate_impact_ratio"],
        "sensitive_SHAP_share": item["sensitive_feature_importance_share"],
    }
    summary_rows.append(row)
display(pd.DataFrame(summary_rows).set_index("variant").round(3))

In [ ]:
for name, item in result["variants"].items():
    print(f"\n{name}")
    print("Counterfactual flip rates:", item["counterfactual_flip_rates"] or "not applicable: protected features omitted")
    display(pd.DataFrame(item["shap_direction_summary"]).head(10).round(4))

## Interpretation

Compare fairness metrics and predictive performance together; a lower fairness gap does not automatically make a model preferable if it substantially reduces utility or creates new group-specific errors. The proxy screen only ranks observed associations, and dropping correlated features may remove useful signal without eliminating all proxy information. Counterfactual tests are model sensitivity checks, not causal claims; changing one attribute can create unrealistic profiles. Record remaining disparities rather than changing audit thresholds to force a pass.